# Week 2: WAXAL, LoRA training and the cross-domain comparison

Settings: **Accelerator → GPU T4 x2**, **Internet → On**, attach the `fin-incl-akan` dataset.

What this runs, top to bottom (about 4–6 hours):
1. Prepare Ashesi (3 dialects) and WAXAL, make speaker-disjoint splits.
2. Zero-shot Whisper on the WAXAL test set (the "before" number for the second domain).
3. A 20-step training smoke test, so a bug costs minutes, not hours.
4. Three LoRA runs with the **same number of steps**: Ashesi only, WAXAL only, both.
   Each is validated on both domains and scored on both test sets.
5. One comparison table, and a zip of everything small.

## 1. Get the code

In [ ]:
REPO_URL = "https://github.com/camycarl01/akan-asr.git"

# Clone once, then just pull: deleting the folder would also delete the
# converted audio in data/.
!test -d /kaggle/working/akan-asr/.git || git clone -q -b master {REPO_URL} /kaggle/working/akan-asr
!git -C /kaggle/working/akan-asr pull -q
%cd /kaggle/working/akan-asr
!pip install -q -e ".[model]" 2>&1 | grep -v "already satisfied" | tail -3
!git log --oneline | head -3
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Prepare the data
Ashesi ≈ 25 min. WAXAL downloads ~4 GB from Hugging Face and decodes MP3s on 4 cores.

In [ ]:
from pathlib import Path

roots = {}
for csv in Path("/kaggle/input").rglob("data.csv"):
    folder = csv.parent.name.lower()
    for dialect, key in [("asante", "asanti"), ("akuapem", "akuapim"), ("fante", "fanti")]:
        if key in folder:
            roots[dialect] = csv.parent.parent
assert len(roots) == 3, f"A dialect is missing: {roots}"

for dialect, root in roots.items():
    !python scripts/prepare_data.py fin_incl --root "{root}" --dialect {dialect} --workers 4
!python scripts/prepare_data.py waxal --workers 4

## 3. Speaker-disjoint splits
Expect ~100% of Ashesi test sentences seen in training, and ~0% for WAXAL.

In [ ]:
!python scripts/make_splits.py data/manifests/*.csv

## 4. Zero-shot baseline on WAXAL
The Ashesi zero-shot numbers came from Week 1 (WER 145.7%).

In [ ]:
!python scripts/run_baseline.py data/splits/waxal_test.csv --model openai/whisper-small

## 5. Training smoke test (~3 min)
Same batch size as the real runs, so out-of-memory errors show up here.

In [ ]:
!python scripts/train_lora.py data/splits/fin_incl_train.csv data/splits/waxal_train.csv \
    --val data/splits/fin_incl_validation.csv data/splits/waxal_validation.csv \
    --limit 320 --max-steps 20 --eval-every 10 --val-utts 32 --run-name smoke

## 6. The three training runs
Equal steps (3,000 × batch 16 = 48,000 clips seen) so the comparison is at equal compute.
The adapter kept is the one with the lowest *mean* validation WER over both domains.
Each run is scored on both test sets right after it trains, so a later crash keeps earlier results.

In [ ]:
STEPS = 3000
VAL = "data/splits/fin_incl_validation.csv data/splits/waxal_validation.csv"
TEST = "data/splits/fin_incl_test.csv data/splits/waxal_test.csv"
RUNS = {
    "ashesi-only": "data/splits/fin_incl_train.csv",
    "waxal-only":  "data/splits/waxal_train.csv",
    "both":        "data/splits/fin_incl_train.csv data/splits/waxal_train.csv",
}
for name, train in RUNS.items():
    print(f"\n========== {name} ==========")
    !python scripts/train_lora.py {train} --val {VAL} --max-steps {STEPS} --run-name {name}
    !python scripts/run_baseline.py {TEST} --adapter adapters/{name}/best

## 7. Comparison

In [ ]:
import pandas as pd

rows = []
for f in sorted(Path("results").glob("*/summary.csv")):
    if f.parent.name == "smoke-best":
        continue
    rows.append(pd.read_csv(f).assign(model=f.parent.name))
table = pd.concat(rows)
print(table[table.subset == "all"].pivot(index="model", columns="test_set", values="WER %").to_string())
print()
print(table.to_string(index=False))
for name in RUNS:
    h = Path(f"adapters/{name}/history.csv")
    if h.exists():
        print(f"\n{name} validation history"); print(pd.read_csv(h).round(3).to_string(index=False))

## 8. Save the small outputs
Download `week2_outputs.zip` from the Output panel.

In [ ]:
!zip -qr /kaggle/working/week2_outputs.zip data/manifests data/splits results adapters && ls -lh /kaggle/working/week2_outputs.zip